# 05. CLIP vs Moment-DETR feasibility probe

Этот notebook сравнивает сохраненные CLIP и Moment-DETR metrics на одной фиксированной 50-query Charades-STA subset. Moment-DETR рассматривается как isolated raw-video feasibility probe, а не как full official benchmark.


In [1]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'results').exists():
    ROOT = ROOT.parent

def read_json(path):
    with open(ROOT / path, 'r', encoding='utf-8') as f:
        return json.load(f)

rows = []
for name in ['clip_w8_s4_mean', 'clip_w16_s8_mean', 'clip_w32_s16_mean', 'clip_w16_s8_max']:
    cfg = read_json(f'results/clip_vs_moment_detr_50/{name}/run_config.json')
    rows.append(cfg['summary'])
mdetr = read_json('results/moment_detr_charades_50/metrics.json')
rows.append({'model': 'Moment-DETR', 'config_name': 'raw_video_checkpoint', **mdetr})
comparison = pd.DataFrame(rows)
cols = ['model', 'config_name', 'evaluated_queries', 'R@1_IoU_0.3', 'R@1_IoU_0.5', 'R@1_IoU_0.7', 'mIoU', 'inference_time_sec']
comparison[cols]


,model,config_name,evaluated_queries,R@1_IoU_0.3,R@1_IoU_0.5,R@1_IoU_0.7,mIoU,inference_time_sec
0,CLIP,clip_w8_s4_mean,50,0.58,0.52,0.32,0.439378,14.223121
1,CLIP,clip_w16_s8_mean,50,0.68,0.36,0.08,0.400772,2.116203
2,CLIP,clip_w32_s16_mean,50,0.54,0.00,0.00,0.285429,2.091613
3,CLIP,clip_w16_s8_max,50,0.56,0.36,0.06,0.337459,2.093376
4,Moment-DETR,raw_video_checkpoint,50,0.44,0.32,0.04,0.263402,17.933695


Moment-DETR использовал `external/moment_detr/run_on_video/moment_detr_ckpt/model_best.ckpt`. Подробности совместимости и ограничений описаны в `docs/moment_detr.md`.


Вывод: CLIP configurations сравниваются с Moment-DETR только на 50 examples. Это подтверждает, что raw-video Moment-DETR path можно подключить к локальным Charades videos, но это не full Moment-DETR reproduction и не benchmark.


Примечание о воспроизводимости: relevant public scripts - `scripts/run_clip_vs_moment_detr.py` и `scripts/run_moment_detr_probe.py`; этот notebook читает только сохраненные public metrics.
